In [ ]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F

StatementMeta(, f0970d65-aa0e-4952-86d0-6f6164036305, 3, Finished, Available, Finished, False)

In [ ]:
# p_load_type = "FULL" | "INCREMENTAL"

# p_config_load_type = "INCREMENTAL"
# p_default_load_type = "FULL"
# p_file_name= "transactions.csv"

load_type = (
    p_config_load_type.strip()
    if p_config_load_type and p_config_load_type.strip() != ""
    else p_default_load_type.strip()
)

file_name = p_file_name

print(f"Current load type: {load_type}")

df_raw_tx = spark.read.option("header", True).option("inferSchema", True) \
    .csv(f'Files/raw_uploads/{file_name}')

StatementMeta(, f0970d65-aa0e-4952-86d0-6f6164036305, 6, Finished, Available, Finished, False)

Current load type: FULL


In [ ]:
# ─────────────────────────────────────────────
# CLEAN / TRANSFORM
# ─────────────────────────────────────────────
df_clean_tx = (
    df_raw_tx
        .filter(F.col("order_id").isNotNull())
        .filter(F.col("customer_id").isNotNull())
        .filter(F.col("product_id").isNotNull())
        .dropDuplicates(["order_id"])
        .withColumn("order_date",  F.to_date(F.col("order_date"), "yyyy-MM-dd"))
        .withColumn("quantity",    F.col("quantity").cast("int"))
        .withColumn("line_amount", F.col("line_amount").cast("double"))
        .filter(F.col("quantity") > 0)
        .filter(F.col("line_amount") > 0)
        .withColumn("updated_at",  F.current_timestamp())
)

df_clean_tx.cache()

target_table = "silver.clean_transactions"
stage_table  = "stage.stg_new_transactions"


# Columns detect changing
compare_cols = ["customer_id", "product_id", "order_date", "quantity", "line_amount"]

StatementMeta(, f0970d65-aa0e-4952-86d0-6f6164036305, 10, Finished, Available, Finished, False)

In [ ]:

#  detect INSERT / UPDATE from batch
def build_stage_from_batch(df_batch, target_table, compare_cols):

    if not spark.catalog.tableExists(target_table):
        return df_batch.withColumn("change_type", F.lit("INSERT"))

    batch_ids = df_batch.select("order_id")

    df_existing = (
        spark.table(target_table)
             .join(batch_ids, on="order_id", how="inner")  # predicate pushdown
    )

    # ── NEW: order_id doesn't exist ──
    df_new = (
        df_batch.alias("s")
        .join(
            df_existing.select("order_id").alias("t"),
            on="order_id",
            how="left_anti"
        )
        .withColumn("change_type", F.lit("INSERT"))
    )

    # ── CHANGED: same order_id but some diff column change ──
    changed_cond = F.lit(False)
    for col in compare_cols:
        changed_cond = changed_cond | (F.col(f"s.{col}") != F.col(f"t.{col}"))

    df_changed = (
        df_batch.alias("s")
        .join(df_existing.alias("t"), on="order_id", how="inner")
        .filter(changed_cond)
        .select(
            F.col("s.order_id"),
            F.col("s.customer_id"),
            F.col("s.product_id"),
            F.col("s.order_date"),
            F.col("s.quantity"),
            F.col("s.line_amount"),
            F.col("s.updated_at"),
        )
        .withColumn("change_type", F.lit("UPDATE"))
    )

    return df_new.unionByName(df_changed)

StatementMeta(, f0970d65-aa0e-4952-86d0-6f6164036305, 8, Finished, Available, Finished, False)

# MAIN LOAD LOGIC

In [ ]:
if load_type == "FULL":

    print("Running FULL LOAD — transactions...")

    # Stage: UPSERT
    df_stage = df_clean_tx.withColumn("change_type", F.lit("UPSERT"))
    (
        df_stage.write
            .mode("append")
            .format("delta")
            .saveAsTable(stage_table)
    )

    print(f"Stage written: {df_stage.count()} rows")

    # Clean: overwrite all
    (
        df_clean_tx.write
            .mode("overwrite")
            .format("delta")
            .saveAsTable(target_table)
    )
    print("FULL LOAD completed")


elif load_type == "INCREMENTAL":

    print("Running INCREMENTAL LOAD — transactions...")

    # ── Step 1: detect diff BEFORE MERGE ──
    df_stage = build_stage_from_batch(df_clean_tx, target_table, compare_cols)

    insert_count  = df_stage.filter(F.col("change_type") == "INSERT").count()
    update_count  = df_stage.filter(F.col("change_type") == "UPDATE").count()
    stage_count   = insert_count + update_count

    print(f"INSERT: {insert_count} | UPDATE: {update_count} | Total: {stage_count}")

    # stage (overwrite each batch) ──
    if stage_count > 0:
        (
            df_stage.write
                .mode("append")
                .format("delta")
                .saveAsTable(stage_table)
        )
        print("Stage OVERWRITE completed")
    else:
        print("No changes detected — stage not updated")

    #  MERGE into clean_transactions ──
    if spark.catalog.tableExists(target_table):

        delta_target = DeltaTable.forName(spark, target_table)

        (
            delta_target.alias("t")
            .merge(
                df_clean_tx.alias("s"),
                "t.order_id = s.order_id"
            )
            .whenMatchedUpdate(set={
                "customer_id": "s.customer_id",
                "product_id":  "s.product_id",
                "order_date":  "s.order_date",
                "quantity":    "s.quantity",
                "line_amount": "s.line_amount",
                "updated_at":  "s.updated_at",
            })
            .whenNotMatchedInsert(values={
                "order_id":    "s.order_id",
                "customer_id": "s.customer_id",
                "product_id":  "s.product_id",
                "order_date":  "s.order_date",
                "quantity":    "s.quantity",
                "line_amount": "s.line_amount",
                "updated_at":  "s.updated_at",
            })
            .execute()
        )
        print("INCREMENTAL MERGE into clean completed")

    else:
        (
            df_clean_tx.write
                .mode("overwrite")
                .format("delta")
                .saveAsTable(target_table)
        )
        print("Initial load into clean completed")

else:
    raise Exception(f"Unsupported load_type: {load_type}")



# CLEAN CACHE
df_clean_tx.unpersist()

StatementMeta(, f0970d65-aa0e-4952-86d0-6f6164036305, 11, Finished, Available, Finished, False)

Running FULL LOAD — transactions...


Stage written: 500 rows


FULL LOAD completed


DataFrame[order_id: string, order_date: date, customer_id: string, product_id: string, quantity: int, line_amount: double, last_modified: timestamp, updated_at: timestamp]